In [3]:
import sys
import os
sys.path.append(os.path.expanduser(r"C:\thesis\code\official_projects\otc-github\open-the-chests"))

## Dataset Mode for `Pattern` – Sampling from Pre-Recorded Traces

This notebook tests and demonstrates the **dataset mode** added to the `Pattern` class.

Previously, `Pattern` only supported *config mode*: events are generated stochastically from Allen-relation instructions at runtime. Dataset mode adds a second path where **pre-recorded event traces are loaded from a CSV file** and randomly sampled instead of generated.

### Two instruction formats — same list-of-dicts structure

The mode is selected by the presence of a `"dataset"` command in the instruction list:

| Command | Mode | Meaning |
|---|---|---|
| `"instantiate"`, Allen relations | Config | Generate events from a temporal logic spec |
| `"dataset"` | Dataset | Load traces from a CSV and sample at runtime |

**Config instruction example:**
```python
[
    {"command": "delay", "parameters": 10},
    {"command": "instantiate", "parameters": ("A", {"bg": "red"}, {"mu": 2, "sigma": 1}), "variable_name": "e1"},
]
```

**Dataset instruction example:**
```python
[
    {"command": "dataset", "parameters": "path/to/activity.csv"},
    {"command": "delay", "parameters": 10},   # optional
    {"command": "noise", "parameters": 0.0},  # optional
]
```

### CSV format

The CSV must follow the format produced by the data generation pipeline:

| Column | Description |
|---|---|
| `unique_activity_key` | Trace identifier (groups rows into one trace) |
| `device_id` | Event type (sensor name) |
| `start_time` | `HH:MM:SS.ffffff` – converted to seconds |
| `end_time` | `HH:MM:SS.ffffff` – converted to seconds |

Times are **normalized to the trace start** (first event starts at 0).

### Setup – Create a Synthetic CSV for Testing

Since we don't want to depend on external data files in this notebook, we generate a small synthetic CSV in a temporary file. It contains three traces of a fake `cook_dinner` activity with two sensor types.

In [4]:
import tempfile
import os

# Write a small synthetic per-activity CSV to a temp file
CSV_CONTENT = """unique_activity_key,device_id,start_time,end_time
file1_cook_dinner_1,floor_kitchen,00:00:01.000000,00:00:05.000000
file1_cook_dinner_1,tap_kitchen,00:00:03.000000,00:00:10.000000
file1_cook_dinner_1,oven,00:00:07.000000,00:00:20.000000
file2_cook_dinner_1,floor_kitchen,00:00:00.500000,00:00:04.000000
file2_cook_dinner_1,tap_kitchen,00:00:02.000000,00:00:08.000000
file2_cook_dinner_1,oven,00:00:06.000000,00:00:18.000000
file3_cook_dinner_1,floor_kitchen,00:00:00.000000,00:00:03.500000
file3_cook_dinner_1,tap_kitchen,00:00:02.500000,00:00:09.000000
file3_cook_dinner_1,door_fridge,00:00:08.000000,00:00:15.000000
file3_cook_dinner_1,oven,00:00:05.000000,00:00:17.000000
"""

# Write to a temp file
tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
tmp.write(CSV_CONTENT)
tmp.close()
DATA_FILE = tmp.name

print(f"Synthetic CSV written to: {DATA_FILE}")
print("\nCSV contents:")
print(CSV_CONTENT)

Synthetic CSV written to: C:\Users\Iva\AppData\Local\Temp\tmprslsihj2.csv

CSV contents:
unique_activity_key,device_id,start_time,end_time
file1_cook_dinner_1,floor_kitchen,00:00:01.000000,00:00:05.000000
file1_cook_dinner_1,tap_kitchen,00:00:03.000000,00:00:10.000000
file1_cook_dinner_1,oven,00:00:07.000000,00:00:20.000000
file2_cook_dinner_1,floor_kitchen,00:00:00.500000,00:00:04.000000
file2_cook_dinner_1,tap_kitchen,00:00:02.000000,00:00:08.000000
file2_cook_dinner_1,oven,00:00:06.000000,00:00:18.000000
file3_cook_dinner_1,floor_kitchen,00:00:00.000000,00:00:03.500000
file3_cook_dinner_1,tap_kitchen,00:00:02.500000,00:00:09.000000
file3_cook_dinner_1,door_fridge,00:00:08.000000,00:00:15.000000
file3_cook_dinner_1,oven,00:00:05.000000,00:00:17.000000



### Test 1 – Creating a Dataset-Mode Pattern

Verify that `Pattern.__init__` correctly detects the `"dataset"` command and sets `instruction_type = "dataset"`.

In [5]:
from openthechests.src.elements.Pattern import Pattern

dataset_instruction = [
    {"command": "dataset",  "parameters": DATA_FILE},
    {"command": "delay",    "parameters": 5.0},
    {"command": "noise",    "parameters": 0.0},
]

dp = Pattern(instruction=dataset_instruction, id=0)

print("instruction_type :", dp.instruction_type)
print("timeout          :", dp.timeout)
print("noise            :", dp.noise)
print("instruction      :", dp.instruction)   # should be []
print("num traces loaded:", len(dp.traces))

assert dp.instruction_type == "dataset"
assert dp.timeout == 5.0
assert dp.instruction == []
assert len(dp.traces) == 3, f"Expected 3 traces, got {len(dp.traces)}"
print("\n✓ All assertions passed.")

instruction_type : dataset
timeout          : 5.0
noise            : 0.0
instruction      : []
num traces loaded: 3

✓ All assertions passed.


### Test 2 – Inspect Loaded Traces

Check that `_load_traces` correctly parses the CSV: times are converted to seconds, normalized to trace start, and each trace is sorted by event end time.

In [6]:
for i, trace in enumerate(dp.traces):
    print(f"\nTrace {i} ({len(trace)} events):")
    for event in trace:
        print(f"  {event}")

# Verify: every trace starts at 0
for i, trace in enumerate(dp.traces):
    min_start = min(e.start for e in trace)
    assert min_start == 0.0, f"Trace {i} does not start at 0 (min start = {min_start})"

# Verify: each trace is sorted by end time
for i, trace in enumerate(dp.traces):
    ends = [e.end for e in trace]
    assert ends == sorted(ends), f"Trace {i} is not sorted by end time"

# Verify: events have empty attributes
for i, trace in enumerate(dp.traces):
    for event in trace:
        assert event.attributes == {}, f"Expected empty attributes, got {event.attributes}"

print("\n✓ All trace validation checks passed.")


Trace 0 (3 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=4.0)
  Event(type='tap_kitchen', attr={}, start=2.0, end=9.0)
  Event(type='oven', attr={}, start=6.0, end=19.0)

Trace 1 (3 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)
  Event(type='tap_kitchen', attr={}, start=1.5, end=7.5)
  Event(type='oven', attr={}, start=5.5, end=17.5)

Trace 2 (4 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)
  Event(type='tap_kitchen', attr={}, start=2.5, end=9.0)
  Event(type='door_fridge', attr={}, start=8.0, end=15.0)
  Event(type='oven', attr={}, start=5.0, end=17.0)

✓ All trace validation checks passed.


### Test 3 – `sample_timeout()` works the same as in config mode

In [7]:
import random
random.seed(42)

samples = [dp.sample_timeout() for _ in range(10)]
print("Sampled timeouts:", [round(s, 3) for s in samples])

assert all(0.0 <= s <= dp.timeout for s in samples), "Timeout samples out of range"
print("\n✓ All sampled timeouts are within [0, timeout].")

Sampled timeouts: [3.197, 0.125, 1.375, 1.116, 3.682, 3.383, 4.461, 0.435, 2.11, 0.149]

✓ All sampled timeouts are within [0, timeout].


### Test 4 – `visualize()` returns gracefully for dataset patterns

In [8]:
dp.visualize()  # Should print a message and return without error

Pattern 0 is a dataset pattern — no instruction graph to visualize.


### Test 5 – Config-mode Pattern still works unchanged

Make sure the existing config-mode path is unaffected by the change.

In [9]:
config_instruction = [
    {"command": "delay",  "parameters": 10},
    {"command": "noise",  "parameters": 0.1},
    {
        "command": "instantiate",
        "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 2, "sigma": 1}),
        "variable_name": "e1"
    },
    {
        "command": "instantiate",
        "parameters": ("B", {"bg": "blue", "fg": "red"}, {"mu": 6, "sigma": 1}),
        "variable_name": "e2"
    },
    {
        "command": "after",
        "parameters": ["e2", "e1"],
        "variable_name": "e2",
        "other": {"gap_dist": {"mu": 4, "sigma": 1}}
    }
]

cp = Pattern(instruction=config_instruction, id=1)

print("instruction_type :", cp.instruction_type)
print("timeout          :", cp.timeout)
print("noise            :", cp.noise)
print("num instructions :", len(cp.instruction))

assert cp.instruction_type == "config"
assert cp.timeout == 10
assert cp.noise == 0.1
assert len(cp.instruction) == 3   # instantiate x2 + after
assert not hasattr(cp, "traces"), "Config pattern should not have a traces attribute"
print("\n✓ Config-mode pattern unaffected.")

instruction_type : config
timeout          : 10
noise            : 0.1
num instructions : 3

✓ Config-mode pattern unaffected.


### Test 6 – Dataset defaults (no delay / noise commands)

Confirm that omitting `"delay"` and `"noise"` falls back to 0.

In [10]:
minimal_instruction = [
    {"command": "dataset", "parameters": DATA_FILE},
]

mp = Pattern(instruction=minimal_instruction, id=2)

print("timeout (default):", mp.timeout)
print("noise   (default):", mp.noise)

assert mp.timeout == 0
assert mp.noise == 0
print("\n✓ Defaults correct.")

timeout (default): 0
noise   (default): 0

✓ Defaults correct.


### Cleanup

In [ ]:
os.unlink(DATA_FILE)
print("Temp file removed.")